
# 02 - Exploratory Data Analysis (EDA) on stations dataset

**Goal:** Understand the stations dataset structure, detect data quality issues, and define initial hypotheses.

**Outputs:**
- Summary statistics and distributions
- Outliers/bad data
- Initial data quality observations

**Next:** Based on findings, define cleaning rules in `02_cleaning_decisions.ipynb`.


### Import Libraries

In [1]:
# Standard library imports
import os
from pathlib import Path

# Third-party imports
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

# Local application imports
from dlgoes.data.stations.clean import join_stations_thresholds
from dlgoes.utils.config import load_config_ns
from dlgoes.utils.seed import set_seed
from dlgoes.utils.config import find_repo_root

repo_root = find_repo_root(Path.cwd())
cfg = load_config_ns(repo_root / 'configs' / 'base.yaml')
set_seed(cfg.seed)


### Read Data

In [2]:
stations_path = os.path.join(cfg.paths.project_root, cfg.paths.data.stations, 'stations_data.csv')
df_stations = pd.read_csv(stations_path, index_col=0, usecols=[
    'CODE', 'ESTACION', 'LON', 'LAT', 'ALT'
])
df_stations.head(2)

,ESTACION,LON,LAT,ALT
CODE,,,,
X47E0D438,ALAMOR,-80.39788,-4.48047,116.0
X47E09732,LA ARDILLA,-80.39014,-4.48956,116.0


In [3]:
df_ths = join_stations_thresholds(cfg)
df_ths.head(2)

,THS_1,THS_2,MIN_1,MIN_2
CODE,,,,
X107131,8.9,17.9,0,0
X109091,3.6,8.3,0,0


In [4]:
df = pd.concat([df_stations, df_ths], axis=1)
df.head(2)

,ESTACION,LON,LAT,ALT,THS_1,THS_2,MIN_1,MIN_2
CODE,,,,,,,,
X47E0D438,ALAMOR,-80.39788,-4.48047,116.0,9.3,10.2,0.0,0.0
X47E09732,LA ARDILLA,-80.39014,-4.48956,116.0,3.1,16.9,0.0,0.0


### Exploration data

In [5]:
df_ths.describe()

,THS_1,THS_2,MIN_1,MIN_2
count,219.000000,219.000000,219.0,219.0
mean,2.383105,10.859817,0.0,0.0
std,5.219172,29.693922,0.0,0.0
min,0.000000,0.000000,0.0,0.0
25%,0.600000,3.650000,0.0,0.0
50%,1.700000,5.400000,0.0,0.0
75%,2.950000,11.100000,0.0,0.0
max,74.500000,305.500000,0.0,0.0


### Analysis plots

In [6]:
fig = px.scatter_mapbox(df[~df['THS_2'].isna()], lat="LAT", lon="LON", hover_data=['ESTACION'],                            
                            color = 'THS_2',
                            zoom=5, height=400)
    
    
fig.update_layout(
mapbox_style="open-street-map")

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

C:\Users\dparedes\AppData\Local\Temp\ipykernel_23592\588940706.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(df[~df['THS_2'].isna()], lat="LAT", lon="LON", hover_data=['ESTACION'],


In [7]:

tmp = df_ths[["THS_1","THS_2"]].melt(
    var_name="Threshold", value_name="Value"
).dropna()

fig = px.violin(
    tmp,
    x="Threshold",
    y="Value",
    box=True,          # agrega boxplot adentro
    points="all",      # puntos (puedes poner "outliers" si hay muchos)
    title="Comparing Thresholds 1 and Threshold 2"
)

fig.update_layout(
    template="plotly_white",
    width=800,
    height=450,
    yaxis_title="Value of the threshold"
)

fig.show()

### Findings

- Variable `UMBRAL` shows historical stats on meteorogical stations, it indicates the percentil 90% and 99%. 
- The thresholds shows the precipitacion data has a maxium value of 305 mm/h, it is near to the 400 mm/h the SENAMHI users indicates
- The precipitation data shows a balanced distrubucion on the Peru

### Next Actions

- Clean the data
- Recalculate the statistics after cleaning.
- Move the finalized cleaning rules to `src/.../cleaning.py`  once they are defined.s
